# Module 3 · Lab 2 — Tools & Memory

We give the agent **tools** (web search + a custom notification), route to them with **conditional edges**, then add **memory** with checkpointing — first in RAM, then persisted to SQLite.

> `uv sync` done in this folder, the **`3_langgraph/.venv`** kernel selected, and your `.env` has `OPENAI_API_KEY` + `SERPER_API_KEY`.

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import Tool
from dotenv import load_dotenv
from IPython.display import Image, display
from pathlib import Path
import gradio as gr

In [ ]:
load_dotenv(override=True)

### (Optional) LangSmith tracing

Want to *see* every step the agent takes? Set up free tracing at https://smith.langchain.com (Settings → API Keys), then add `LANGCHAIN_API_KEY` and `LANGCHAIN_TRACING_V2=true` to your `.env`. Not required to run the lab.

### Tool 1 — web search (Serper)

LangChain gives us a ready-made web-search wrapper. It reads `SERPER_API_KEY` from your `.env`.

In [ ]:
serper = GoogleSerperAPIWrapper()
serper.run("What is the capital of France?")

### Wrap it as a Tool

A Tool = a name + a function + a description. The description is what the model reads to decide when to use it.

In [ ]:
tool_search = Tool(
    name="search",
    func=serper.run,
    description="Useful when you need up-to-date information from an online search",
)
tool_search.invoke("What is the capital of France?")

### Tool 2 — our own custom tool

Any Python function can be a tool. Ours "notifies" the user by appending the message to a local file, `output/notifications.md` — no accounts or app to install.

In [ ]:
def notify(text: str) -> str:
    """Send the user a notification (writes to output/notifications.md)."""
    Path("output").mkdir(exist_ok=True)
    with open("output/notifications.md", "a", encoding="utf-8") as f:
        f.write(f"- {text}\n")
    return "notification sent"

tool_push = Tool(
    name="send_notification",
    func=notify,
    description="Send the user a notification with a short message",
)
tool_push.invoke("Hello, me")

### Bring the tools together

When an LLM uses tools there are always two jobs: **(1)** tell the model what tools exist, and **(2)** actually run a tool when it asks. LangGraph has a home for each.

In [ ]:
tools = [tool_search, tool_push]

### Step 1 — State

This time a `TypedDict` (a dict with declared keys) — same `add_messages` reducer.

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
graph_builder = StateGraph(State)

### Job 1 — `bind_tools`

`bind_tools` makes a version of the LLM that knows about our tools and can ask to call them.

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")
llm_with_tools = llm.bind_tools(tools)

### Nodes — the chatbot, and the tool runner

The `chatbot` node calls the tool-aware LLM. The `tools` node is a pre-built `ToolNode` that actually runs whichever tool the model asked for (Job 2).

In [ ]:
def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

### Edges — a conditional edge

`tools_condition` checks whether the model asked for a tool. If so, route to the tools node; its result loops back to the chatbot to finish the answer.

In [ ]:
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [ ]:
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### Run it

Try: *search the USD to AED exchange rate and notify me*. Then open `output/notifications.md`. But tell it your name and ask again — still no memory.

In [ ]:
def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]})
    return result["messages"][-1].content

gr.ChatInterface(chat, type="messages").launch()

## Add memory — checkpointing

Why doesn't it remember? Because one `invoke` = one **super-step** (the whole graph runs once). The reducer combines state *within* a super-step, not *between* them.

To carry memory across turns, compile with a **checkpointer** and pass a **`thread_id`**.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
# Same graph — the only change is checkpointer=memory at compile
graph_builder = StateGraph(State)

llm = ChatOpenAI(model="gpt-5.4-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph = graph_builder.compile(checkpointer=memory)

### Now it remembers

`thread_id` picks which conversation. Tell it your name, then ask — it knows.

In [ ]:
config = {"configurable": {"thread_id": "1"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content

gr.ChatInterface(chat, type="messages").launch()

### Peek at the saved state (time travel)

Every super-step is checkpointed — you can read the current snapshot and the full history, and even rewind to a past checkpoint.

In [ ]:
graph.get_state(config)

In [ ]:
# Most recent first
list(graph.get_state_history(config))

## Persist it — SQLite

`MemorySaver` lives in RAM (gone on restart). Swap in `SqliteSaver` and the conversation survives a full restart — stored in `memory.db`. Just one line changes.

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

conn = sqlite3.connect("memory.db", check_same_thread=False)
sql_memory = SqliteSaver(conn)

In [ ]:
graph_builder = StateGraph(State)

llm = ChatOpenAI(model="gpt-5.4-mini")
llm_with_tools = llm.bind_tools(tools)

def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_conditional_edges("chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

graph = graph_builder.compile(checkpointer=sql_memory)

In [ ]:
config = {"configurable": {"thread_id": "3"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content

gr.ChatInterface(chat, type="messages").launch()

## That's LangGraph!

You built a graph, gave it tools with conditional routing, and added memory you can persist and replay.

**Lab 3 is yours** — pick a project, sketch the graph, and let Codex build it within what these labs taught (see `AGENTS.md`).